# 04 - Auditoria de 30 dias e mineracao de padroes

Usa somente eventos com comportamento confirmado/corrigido por humano e o catalogo Lean. Esta e uma **proxy de desenvolvimento**, nao o holdout binario final. O notebook bloqueia qualquer arquivo com `gabarito` no nome.

In [ ]:
from pathlib import Path
import json, os, sys
import pandas as pd

ROOT = Path.cwd()
NB_DIR = ROOT / 'notebooks' if (ROOT / 'notebooks').is_dir() else Path.cwd()
sys.path.insert(0, str(NB_DIR))
from produtividade_30d import (carregar_fontes, preparar_dataset, dividir_por_dia,
    resumo_dataset, tabela_por_dia, minerar_categorias, minerar_ngramas)

OUT = Path(os.environ.get('KV_30D_OUTPUT_DIR', NB_DIR / 'outputs' / 'productivity_30d'))
OUT.mkdir(parents=True, exist_ok=True)
eventos, catalogo, fontes = carregar_fontes()
proxy, populacao = preparar_dataset(eventos, catalogo)
split = dividir_por_dia(proxy)
resumo = resumo_dataset(proxy, populacao)
resumo['dias_treino'] = list(split.dias_treino)
resumo['dias_calibracao'] = list(split.dias_calibracao)
resumo['dias_teste_interno'] = list(split.dias_teste)
print(json.dumps(resumo, indent=2, ensure_ascii=False))

diario = tabela_por_dia(proxy)
padroes = minerar_categorias(split.treino)
ngramas = minerar_ngramas(split.treino)
(OUT / 'dataset_summary.json').write_text(json.dumps(resumo, indent=2, ensure_ascii=False), encoding='utf-8')
diario.to_csv(OUT / 'metricas_por_dia.csv', index=False)
padroes.to_csv(OUT / 'padroes_categoricos_treino.csv', index=False)
ngramas.to_csv(OUT / 'padroes_ngramas_treino.csv', index=False)
display(diario)
display(padroes.head(20))
display(pd.concat([ngramas.head(12), ngramas.tail(12)]))